# 07. Real checkpoints on a Colab GPU (optional)

**Goal:** run the plain-versus-warm-up comparison on the real fine-tuned checkpoints and real CT, instead of reading it off a CSV.

**This notebook is optional and additive.** It needs a Colab **GPU** runtime and the weights; 01-06 stay laptop-runnable with no GPU and are unaffected.

**Inference only. No training happens here.**

The two checkpoints are fold-0, 1000-epoch fine-tunes of the **MultiTalentV2 Challenge Edition** pretrained CT model (Ulrich, DKFZ; https://zenodo.org/records/13753413) on `Dataset591_liver_lesions`, differing only in the LR schedule:

| Variant | Schedule | Liver-lesion Dice (169 validation cases) |
|---|---|---|
| `plain1e3` | LR 1e-3, default PolyLR | 0.7890 |
| `warmup1e3` | 50-epoch linear warm-up to 1e-3, then offset PolyLR | 0.8027 |

Runtime: **Runtime > Change runtime type > GPU**, then run top to bottom.

> These checkpoints were trained on a 20 GB GPU slice. Inference needs less memory than training, so a 16 GB Colab T4 should be enough - but that has not been measured yet. If you hit out-of-memory, cell 5 has a fallback ladder.

## Data attribution

The CT cases used below come from **`Dataset591_liver_lesions`** - "Training dataset for TotalSegmentator task liver_lesions", Jakob Wasserthal, University Hospital Basel: https://doi.org/10.5281/zenodo.20272572

Only a couple of held-out cases are used here; nothing is committed to the course repo.

## 1. Guard: is there actually a GPU?

Without CUDA, ResEnc-L `3d_fullres` sliding-window inference at 1 mm isotropic will grind for a very long time. Fail loudly and early instead.

In [ ]:
import shutil, subprocess, sys

HAS_GPU = False
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                          '--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip())
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
    print('torch', torch.__version__, '| cuda available:', HAS_GPU)
except ImportError:
    print('torch not importable yet - it is installed in the next cell')

if not HAS_GPU:
    print('\n*** NO GPU DETECTED ***')
    print('Colab: Runtime > Change runtime type > Hardware accelerator > GPU, then rerun.')
    print('You can still finish the course without this notebook: 01-06 are CPU-only.')

## 2. Install nnU-Net (the version pin is load-bearing)

An nnU-Net checkpoint stores the *identity* of its trainer class and plans. Load it with a different nnU-Net version and you get an unhelpful key or attribute error rather than a clear "version mismatch". `NNUNET_VERSION` below is the version the checkpoints were trained with - change it only if you supply your own weights.

Colab ships its own torch build, so this install may need a **runtime restart** (*Runtime > Restart session*). If prompted, restart and rerun from cell 1; the env vars are re-set below, so nothing is lost.

In [ ]:
NNUNET_VERSION = '2.6.4'   # the version these checkpoints were trained with

get_ipython().system(f'pip install -q nnunetv2=={NNUNET_VERSION}')

import os
from pathlib import Path

ROOT = Path('/content/nnunet') if Path('/content').exists() else Path.cwd() / 'nnunet'
RAW, PREP, RESULTS = ROOT / 'nnUNet_raw', ROOT / 'nnUNet_preprocessed', ROOT / 'nnUNet_results'
for d in (RAW, PREP, RESULTS):
    d.mkdir(parents=True, exist_ok=True)
os.environ['nnUNet_raw'] = str(RAW)
os.environ['nnUNet_preprocessed'] = str(PREP)
os.environ['nnUNet_results'] = str(RESULTS)
print('nnUNet_results =', RESULTS)

## 3. Fetch the two checkpoints

nnU-Net finds a model by directory layout, not by argument, so the files must land at

```
$nnUNet_results/Dataset591_liver_lesions/<trainer>__<plans>__3d_fullres/
    plans.json
    dataset.json
    fold_0/checkpoint_final.pth
```

`CKPT_SOURCE` is the only thing you need to change to use **your own** fine-tunes: point it at a Hugging Face repo id, or set `LOCAL_CKPT_DIR` to a directory (or mounted Drive path) that already has the layout above and the download is skipped entirely.

Each file is about 820 MB, so this cell takes a few minutes. Most of that is optimizer state nnU-Net saves for resuming training, which inference does not need - the checkpoints are published unmodified so their sha256 sums (in `MODEL_CARD.md`) match the originals.

In [ ]:
# --- configure the weight source -------------------------------------------------
CKPT_SOURCE = 'KS987/multitalentv2-finetune-liver'   # Hugging Face repo id
LOCAL_CKPT_DIR = None     # or point this at your own fine-tunes in $nnUNet_results layout

DATASET = 'Dataset591_liver_lesions'
PLANS = 'nnUNetResEncUNetL1x1x1_Plans_znorm_bs24_mig_bs1'
VARIANTS = {
    'plain 1e-3 (no warm-up)': 'nnUNetTrainer_plain1e3_wandb',
    'warm-up to 1e-3':         'nnUNetTrainer_warmup1e3_wandb',
}
MODEL_DIRS = {label: RESULTS / DATASET / f'{t}__{PLANS}__3d_fullres'
              for label, t in VARIANTS.items()}

if LOCAL_CKPT_DIR:
    src = Path(LOCAL_CKPT_DIR)
    print('using local checkpoints from', src)
    shutil.copytree(src, RESULTS / DATASET, dirs_exist_ok=True)
elif CKPT_SOURCE:
    get_ipython().system('pip install -q huggingface_hub')
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=CKPT_SOURCE, allow_patterns=[f'{DATASET}/**'],
                      local_dir=str(RESULTS))
else:
    raise SystemExit(
        'No weight source configured.\n'
        'Set CKPT_SOURCE to the published Hugging Face repo id, or LOCAL_CKPT_DIR to '
        'your own fine-tunes in $nnUNet_results layout. See README for status.'
    )

for label, d in MODEL_DIRS.items():
    ck = d / 'fold_0' / 'checkpoint_final.pth'
    print(f'{label:26s} {ck.stat().st_size / 1e6:7.1f} MB  {ck.exists()}')

## 3b. Register the two trainer names

nnU-Net resolves a checkpoint's trainer **by class name**, searching the installed `nnunetv2` package. `nnUNetTrainer_plain1e3_wandb` and `nnUNetTrainer_warmup1e3_wandb` are custom variants that live in the lab's training environment, not in the pip package, so `nnUNetv2_predict` would fail with a "could not find trainer" error until they exist here too.

The LR schedule is a *training-time* concern and has no effect on inference, so plain subclasses are enough to make the names resolve and build the identical network. This is the practical cost of the version/trainer identity baked into an nnU-Net checkpoint - worth seeing rather than hiding.

In [ ]:
import nnunetv2, textwrap

variants = Path(nnunetv2.__file__).parent / 'training' / 'nnUNetTrainer' / 'variants' / 'lr_schedule'
variants.mkdir(parents=True, exist_ok=True)
(variants / '__init__.py').touch()
for name in ('plain1e3', 'warmup1e3'):
    (variants / f'{name}_wandb.py').write_text(textwrap.dedent(f'''
        from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer


        class nnUNetTrainer_{name}_wandb(nnUNetTrainer):
            """Inference-only shim: the real variant differs in LR schedule."""
    ''').lstrip())
print('registered:', [p.name for p in sorted(variants.glob('*1e3_wandb.py'))])

## 4. Fetch the CT cases

nnU-Net's raw naming is `<CASE>_0000.nii.gz` for images (`_0000` is the modality index, here the single CT channel) and `<CASE>.nii.gz` for labels. Two cases by default.

In [ ]:
IMAGES, LABELS = RAW / DATASET / 'imagesTs', RAW / DATASET / 'labelsTs'
for d in (IMAGES, LABELS):
    d.mkdir(parents=True, exist_ok=True)

if CKPT_SOURCE:
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=CKPT_SOURCE, allow_patterns=['cases/**'],
                      local_dir=str(ROOT / 'download'))
    for f in sorted((ROOT / 'download' / 'cases').rglob('*.nii.gz')):
        (LABELS if '_0000' not in f.name else IMAGES).joinpath(f.name).write_bytes(f.read_bytes())

cases = sorted(p.name.replace('_0000.nii.gz', '') for p in IMAGES.glob('*_0000.nii.gz'))
print('cases:', cases)
print('\nData: Dataset591_liver_lesions (Wasserthal, University Hospital Basel), '
      'doi:10.5281/zenodo.20272572')

## 5. Predict twice - once per schedule

Same data, same plans, same architecture; the *only* difference between the two runs is which fine-tuned weights are loaded.

`--disable_tta` turns off test-time augmentation (8x mirroring). It is the first rung of the VRAM/time ladder and costs a little accuracy; if you still hit OOM, raise `--step_size` (fewer sliding-window positions), and as a last resort add `-device cpu` - correct, but expect tens of minutes per case.

In [ ]:
import time

PRED_DIRS = {}
for label, model_dir in MODEL_DIRS.items():
    out = ROOT / 'pred' / model_dir.name
    out.mkdir(parents=True, exist_ok=True)
    PRED_DIRS[label] = out
    t0 = time.perf_counter()
    get_ipython().system(
        f'nnUNetv2_predict -i {IMAGES} -o {out} -d {DATASET} -c 3d_fullres -f 0 '
        f'-tr {VARIANTS[label]} -p {PLANS} -chk checkpoint_final.pth --disable_tta'
    )
    print(f'{label:26s} {time.perf_counter() - t0:6.1f} s wall clock')

## 6. Score both against the ground truth

Reusing `course_utils.dsc` - the same Dice implementation NB06 verified against hand-built cases. Label 1 is the liver lesion.

In [ ]:
get_ipython().system('pip install -q nibabel')
import nibabel as nib
import numpy as np

# course_utils is a two-file dependency; fetch it if we are on a bare Colab runtime.
try:
    from course_utils.dsc import dice_for_label
except ImportError:
    get_ipython().system('git clone -q https://github.com/Kappapapa123/internal-mini-course /content/course')
    sys.path.insert(0, '/content/course')
    from course_utils.dsc import dice_for_label

def load(p):
    return np.rint(np.asarray(nib.load(str(p)).dataobj)).astype(int)

scores = {}
for label, out in PRED_DIRS.items():
    scores[label] = [dice_for_label(load(out / f'{c}.nii.gz'),
                                    load(LABELS / f'{c}.nii.gz'), 1) for c in cases]

print(f"{'variant':26s} " + '  '.join(f'{c:>12s}' for c in cases) + '    mean')
for label, s in scores.items():
    print(f'{label:26s} ' + '  '.join(f'{v:12.4f}' for v in s) + f'  {np.nanmean(s):6.4f}')

delta = np.nanmean(scores['warm-up to 1e-3']) - np.nanmean(scores['plain 1e-3 (no warm-up)'])
print(f'\nwarm-up minus plain, on these {len(cases)} cases: {delta:+.4f} Dice')
print('for reference, over all 169 fold-0 validation cases: '
      '0.7890 plain vs 0.8027 warm-up  (+0.0137)')

## 7. Look at the segmentations

A Dice number on two cases hides a lot. Overlay ground truth against both predictions on the slice with the most lesion voxels.

In [ ]:
import matplotlib.pyplot as plt
try:
    from course_utils.viz import overlay_mask_on_slice
except ImportError:
    sys.path.insert(0, '/content/course')
    from course_utils.viz import overlay_mask_on_slice

for c in cases:
    ct = np.asarray(nib.load(str(IMAGES / f'{c}_0000.nii.gz')).dataobj, dtype=np.float32)
    gt = load(LABELS / f'{c}.nii.gz')
    z = int(np.argmax((gt == 1).sum(axis=(0, 1))))   # busiest lesion slice
    panels = [('ground truth', gt)] + [(lab, load(PRED_DIRS[lab] / f'{c}.nii.gz'))
                                       for lab in PRED_DIRS]
    fig, axes = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4))
    for ax, (title, mask) in zip(axes, panels):
        overlay_mask_on_slice(ct, mask, z=z, ax=ax)
        ax.set_title(title)
    fig.suptitle(f'{c}  (z={z})')
    fig.tight_layout()
    plt.show()

## Recap, and three honesty notes

You just ran the course's central claim end to end on real weights and real CT: same pretrained checkpoint, same data, same plans, two LR schedules, and a measurable Dice difference. Three things this notebook does **not** show:

1. **A couple of cases is an anecdote, not an evaluation.** Whatever gap you got above, the number to quote is the one over all 169 fold-0 validation cases: **0.789 plain vs 0.803 warm-up**. Two cases can easily land the wrong way round. And even that 169-case number is one fold of one dataset, with no error bar.
2. **The training curves in NB03 do not predict this gap.** `assets/precomputed/real_training_curves.csv` is these same two runs, and its curves nearly overlap - peaking at 0.899 plain vs 0.902 warm-up, ending at 0.882 vs 0.881. That is nnU-Net's *mean-foreground pseudo-Dice* during training, a different metric from the *liver-lesion-class* Dice at final evaluation. Same runs, different metric: the separation shows up in per-class evaluation, not in the curve.
3. **The gain is task dependent.** +1.37 Dice points on liver lesions here, about +0.4 on TBI, and no benefit at all on PANTHER. Quote the 591 numbers for 591. Run the small plain-versus-warm-up ablation on *your* task before assuming warm-up helps - that is the actual lesson of this course.